# Dimension Row Estimation — STATS19 Data Warehouse (12 Dimensions)

**Mục tiêu**: Profiling ước tính số dòng dự kiến cho **toàn bộ 12 bảng Dimension** trong schema Snowflake.

**Nguồn dữ liệu**: Đọc từ file **Clean** (`Processed/`). KHÔNG đọc file Raw.

## Phương pháp
- **Distinct rows** = Số tổ hợp DISTINCT của các cột grain có trong dữ liệu thực tế.
- **Unknown row** = Kiểm tra trực tiếp trong data: nếu có bất kỳ tổ hợp nào chứa giá trị `'Unknown'` hoặc `NaN` thì gán bằng 1 (vì khi load vào bảng Dim chỉ dùng 1 dòng placeholder Unknown duy nhất), ngược lại bằng 0.
- **Expected rows** = Distinct rows + Unknown row.

**Công thức tổng quát**:
```
Expected Rows = |DISTINCT(grain_columns)| + (1 nếu data có chứa Unknown/NaN ngược lại 0)
```

## Danh sách 12 Dimension

| # | Dimension | Loại | Nguồn file | Grain |
|---|---|---|---|---|
| 1 | DimDate | Trực tiếp | collisions_clean.csv | collision_date |
| 2 | DimTimeBand | Phụ (Snowflake) | collisions_clean.csv | time_band (derived từ hour) |
| 3 | DimTime | Trực tiếp | collisions_clean.csv | collision_time (HH:MM) |
| 4 | DimLocalAuthority | Phụ (Snowflake) | collisions_clean.csv | local_authority_district_code |
| 5 | DimLocation | Trực tiếp | collisions_clean.csv | longitude, latitude, local_authority_district_code |
| 6 | DimRoad | Trực tiếp | collisions_clean.csv | first_road_class, road_type, speed_limit_mph, area_type |
| 7 | DimEnvironment | Trực tiếp (Junk) | collisions_clean.csv | light_conditions, weather_conditions, road_surface_conditions |
| 8 | DimJunction | Trực tiếp | collisions_clean.csv | junction_detail, junction_control, pedestrian_crossing |
| 9 | DimSeverity | Trực tiếp (Role-playing) | collisions + casualties | UNION(collision_severity, casualty_severity) |
| 10 | DimPedestrianContext | Phụ (Snowflake) | casualties_clean.csv | pedestrian_location, pedestrian_movement |
| 11 | DimCasualtyProfile | Trực tiếp | casualties_clean.csv | casualty_age_band, casualty_sex, casualty_class, casualty_type, pedestrian_location, pedestrian_movement |
| 12 | DimVehicleProfile | Trực tiếp | vehicles_clean.csv | vehicle_type, driver_age_band, driver_sex, vehicle_manoeuvre, skidding_and_overturning, vehicle_leaving_carriageway |


In [61]:
# CELL 1 — Import và cấu hình đường dẫn

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

BASE_DIR = Path(r"D:/OLAP/Data/Processed")

COLLISIONS_FILE = BASE_DIR / "collisions_clean.csv"
VEHICLES_FILE = BASE_DIR / "vehicles_clean.csv"
CASUALTIES_FILE = BASE_DIR / "casualties_clean.csv"

OUTPUT_DIR = Path(r"D:/OLAP/Data/Profiling")
OUTPUT_FILE = OUTPUT_DIR / "dimension_row_estimation_report.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for file_path in [
    COLLISIONS_FILE,
    VEHICLES_FILE,
    CASUALTIES_FILE,
]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file: {file_path}"
        )

print("Đã tìm thấy đầy đủ ba file clean.")


Đã tìm thấy đầy đủ ba file clean.


In [62]:
import pandas as pd

df_col = pd.read_csv(
    COLLISIONS_FILE,
    dtype={"collision_id": "string"},
    low_memory=False
)

df_veh = pd.read_csv(
    VEHICLES_FILE,
    dtype={
        "collision_id": "string",
        "vehicle_reference": "Int64"
    },
    low_memory=False
)

df_cas = pd.read_csv(
    CASUALTIES_FILE,
    dtype={
        "collision_id": "string",
        "vehicle_reference": "Int64",
        "casualty_reference": "Int64"
    },
    low_memory=False
)

print("Collisions:", len(df_col))
print("Vehicles:", len(df_veh))
print("Casualties:", len(df_cas))

print(
    "Duplicate collision:",
    df_col.duplicated(["collision_id"]).sum()
)
print(
    "Duplicate vehicle:",
    df_veh.duplicated(
        ["collision_id", "vehicle_reference"]
    ).sum()
)
print(
    "Duplicate casualty:",
    df_cas.duplicated(
        ["collision_id", "vehicle_reference", "casualty_reference"]
    ).sum()
)

Collisions: 513801
Vehicles: 937265
Casualties: 652821
Duplicate collision: 0
Duplicate vehicle: 0
Duplicate casualty: 0


In [63]:
# CELL 2 — Load dữ liệu từ file Clean

COLLISIONS_COLS = [
    "collision_id",
    "collision_year",
    "collision_date",
    "collision_time",
    "longitude",
    "latitude",
    "local_authority_district_code",
    "collision_severity",
    "first_road_class",
    "road_type",
    "speed_limit_mph",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "area_type",
]

VEHICLES_COLS = [
    "collision_id",
    "vehicle_reference",
    "vehicle_type",
    "vehicle_manoeuvre",
    "driver_age_band",
    "driver_sex",
    "skidding_and_overturning",
    "vehicle_leaving_carriageway",
]

CASUALTIES_COLS = [
    "collision_id",
    "vehicle_reference",
    "casualty_reference",
    "casualty_class",
    "casualty_sex",
    "casualty_age",
    "casualty_age_band",
    "casualty_severity",
    "casualty_type",
    "pedestrian_location",
    "pedestrian_movement",
]

print("Đang đọc collisions_clean.csv...")
df_col = pd.read_csv(
    COLLISIONS_FILE,
    usecols=COLLISIONS_COLS,
    dtype=str,
    low_memory=False,
)

print("Đang đọc vehicles_clean.csv...")
df_veh = pd.read_csv(
    VEHICLES_FILE,
    usecols=VEHICLES_COLS,
    dtype=str,
    low_memory=False,
)

print("Đang đọc casualties_clean.csv...")
df_cas = pd.read_csv(
    CASUALTIES_FILE,
    usecols=CASUALTIES_COLS,
    dtype=str,
    low_memory=False,
)

print("\nSỐ DÒNG ĐẦU VÀO")
print(f"Collisions : {len(df_col):,}")
print(f"Vehicles   : {len(df_veh):,}")
print(f"Casualties : {len(df_cas):,}")

Đang đọc collisions_clean.csv...
Đang đọc vehicles_clean.csv...
Đang đọc casualties_clean.csv...

SỐ DÒNG ĐẦU VÀO
Collisions : 513,801
Vehicles   : 937,265
Casualties : 652,821


In [64]:
fact_vehicle_count = (
    df_cas[
        ["collision_id", "vehicle_reference"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Số phương tiện phân biệt thực sự được "
    f"FactCasualty tham chiếu: {fact_vehicle_count:,}"
)

Số phương tiện phân biệt thực sự được FactCasualty tham chiếu: 566,945


In [65]:
# CELL 3 Chuẩn hóa cách nhận diện Unknown
UNKNOWN_EXACT_VALUES = {
    "",
    "unknown",
    "not known",
    "not reported",
    "data missing or out of range",
    "unallocated",
    "nan",
    "<na>",
    "none",
}


def normalized_text(series):
    """
    Chuẩn hóa khoảng trắng để phục vụ profiling.
    Không thay đổi các nhãn nghiệp vụ hợp lệ.
    """
    return series.astype("string").str.strip()


def is_unknown(series):
    """
    Trả về True nếu giá trị là missing hoặc mang ý nghĩa Unknown.
    """
    text = normalized_text(series).str.casefold()

    return (
        series.isna()
        | text.isin(UNKNOWN_EXACT_VALUES)
        | text.str.startswith("unknown", na=False)
    )


def canonicalize_unknown(series):
    """
    Chuyển mọi cách biểu diễn missing/unknown về chuỗi 'Unknown'.
    """
    result = normalized_text(series)
    result = result.mask(is_unknown(result), "Unknown")
    return result

In [66]:
# Cell 4 — Chuẩn hóa các cột phân loại trước khi đếm

COLLISION_LABEL_COLS = [
    "local_authority_district_code",
    "collision_severity",
    "first_road_class",
    "road_type",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "area_type",
]

VEHICLE_LABEL_COLS = [
    "vehicle_type",
    "vehicle_manoeuvre",
    "driver_age_band",
    "driver_sex",
    "skidding_and_overturning",
    "vehicle_leaving_carriageway",
]

CASUALTY_LABEL_COLS = [
    "casualty_class",
    "casualty_sex",
    "casualty_age_band",
    "casualty_severity",
    "casualty_type",
    "pedestrian_location",
    "pedestrian_movement",
]

for column in COLLISION_LABEL_COLS:
    df_col[column] = canonicalize_unknown(df_col[column])

for column in VEHICLE_LABEL_COLS:
    df_veh[column] = canonicalize_unknown(df_veh[column])

for column in CASUALTY_LABEL_COLS:
    df_cas[column] = canonicalize_unknown(df_cas[column])

print("Đã chuẩn hóa cách biểu diễn Unknown.")

Đã chuẩn hóa cách biểu diễn Unknown.


In [67]:
# Cell 5 — Hàm profiling
def profile_standard_dimension(
    dimension_name,
    dataframe,
    grain_columns,
    source_file,
    note="",
):
    """
    Quy tắc:

    1. Đếm các tổ hợp DISTINCT theo grain.
    2. Giữ lại các tổ hợp chỉ thiếu một phần thông tin.
    3. Nếu tổ hợp có toàn bộ thuộc tính Unknown,
       tổ hợp đó được thay bằng dòng kỹ thuật Key = 0.
    4. Mỗi Dimension luôn có đúng một dòng kỹ thuật Unknown.
    """

    distinct_df = (
        dataframe[grain_columns]
        .drop_duplicates()
        .copy()
    )

    unknown_matrix = pd.DataFrame(
        {
            column: is_unknown(distinct_df[column])
            for column in grain_columns
        },
        index=distinct_df.index,
    )

    any_unknown_mask = unknown_matrix.any(axis=1)
    all_unknown_mask = unknown_matrix.all(axis=1)

    partial_unknown_mask = (
        any_unknown_mask & ~all_unknown_mask
    )

    source_distinct_rows = len(distinct_df)
    all_unknown_combinations = int(all_unknown_mask.sum())
    partial_unknown_combinations = int(
        partial_unknown_mask.sum()
    )

    # Bỏ tổ hợp toàn Unknown vì nó sẽ được đại diện bởi Key = 0
    valid_members = int((~all_unknown_mask).sum())

    # Mỗi Dimension luôn có một dòng kỹ thuật Unknown
    technical_members = 1

    expected_rows = valid_members + technical_members

    return {
        "Dimension": dimension_name,
        "Source File": source_file,
        "Grain": ", ".join(grain_columns),
        "Total Source Rows": len(dataframe),
        "Source Distinct Rows": source_distinct_rows,
        "Partial-Unknown Combinations":
            partial_unknown_combinations,
        "All-Unknown Combinations":
            all_unknown_combinations,
        "Valid Members": valid_members,
        "Technical Members": technical_members,
        "Expected Rows": expected_rows,
        "Note": note,
    }


all_results = []

print("Hàm profiling Dimension đã sẵn sàng.")

Hàm profiling Dimension đã sẵn sàng.


In [68]:
# CELL 6 — DimDate

result = profile_standard_dimension(
    dimension_name="DimDate",
    dataframe=df_col,
    grain_columns=["collision_date"],
    source_file="collisions_clean.csv",
    note="Một dòng cho một ngày; có thêm Unknown Date.",
)

all_results.append(result)
display(pd.DataFrame([result]))


,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimDate,collisions_clean.csv,collision_date,513801,1826,0,0,1826,1,1827,Một dòng cho một ngày; có thêm Unknown Date.


In [69]:
# CELL 7 — DimTimeBand
TIME_BANDS = pd.DataFrame(
    [
        {
            "TimeBandKey": 1,
            "TimeBandName": "Overnight",
            "TimeBandSort": 1,
            "StartHour": 0,
            "EndHour": 4,
        },
        {
            "TimeBandKey": 2,
            "TimeBandName": "Early Morning",
            "TimeBandSort": 2,
            "StartHour": 5,
            "EndHour": 6,
        },
        {
            "TimeBandKey": 3,
            "TimeBandName": "Morning Rush",
            "TimeBandSort": 3,
            "StartHour": 7,
            "EndHour": 9,
        },
        {
            "TimeBandKey": 4,
            "TimeBandName": "Working Hours",
            "TimeBandSort": 4,
            "StartHour": 10,
            "EndHour": 15,
        },
        {
            "TimeBandKey": 5,
            "TimeBandName": "Evening Rush",
            "TimeBandSort": 5,
            "StartHour": 16,
            "EndHour": 18,
        },
        {
            "TimeBandKey": 6,
            "TimeBandName": "Evening",
            "TimeBandSort": 6,
            "StartHour": 19,
            "EndHour": 22,
        },
        {
            "TimeBandKey": 7,
            "TimeBandName": "Late Night",
            "TimeBandSort": 7,
            "StartHour": 23,
            "EndHour": 23,
        },
    ]
)

result = {
    "Dimension": "DimTimeBand",
    "Source File": "Business-defined",
    "Grain": "TimeBandName",
    "Total Source Rows": len(df_col),
    "Source Distinct Rows": 7,
    "Partial-Unknown Combinations": 0,
    "All-Unknown Combinations": 0,
    "Valid Members": 7,
    "Technical Members": 1,
    "Expected Rows": 8,
    "Note": "7 khung giờ nghiệp vụ + 1 dòng Unknown.",
}

all_results.append(result)

display(TIME_BANDS)
display(pd.DataFrame([result]))

,TimeBandKey,TimeBandName,TimeBandSort,StartHour,EndHour
0,1,Overnight,1,0,4
1,2,Early Morning,2,5,6
2,3,Morning Rush,3,7,9
3,4,Working Hours,4,10,15
4,5,Evening Rush,5,16,18
5,6,Evening,6,19,22
6,7,Late Night,7,23,23


,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimTimeBand,Business-defined,TimeBandName,513801,7,0,0,7,1,8,7 khung giờ nghiệp vụ + 1 dòng Unknown.


In [70]:
# Cell 8 — Hàm xác định TimeBand
def assign_time_band(time_value):
    if pd.isna(time_value):
        return "Unknown"

    text = str(time_value).strip()

    try:
        hour = int(text.split(":")[0])
    except (ValueError, IndexError):
        return "Unknown"

    if 0 <= hour <= 4:
        return "Overnight"
    if 5 <= hour <= 6:
        return "Early Morning"
    if 7 <= hour <= 9:
        return "Morning Rush"
    if 10 <= hour <= 15:
        return "Working Hours"
    if 16 <= hour <= 18:
        return "Evening Rush"
    if 19 <= hour <= 22:
        return "Evening"
    if hour == 23:
        return "Late Night"

    return "Unknown"


time_band_check = (
    df_col["collision_time"]
    .apply(assign_time_band)
    .value_counts(dropna=False)
    .rename_axis("TimeBand")
    .reset_index(name="CollisionCount")
)

display(time_band_check)

,TimeBand,CollisionCount
0,Working Hours,184238
1,Evening Rush,123623
2,Evening,79889
3,Morning Rush,75998
4,Overnight,24388
5,Early Morning,14359
6,Late Night,11306


In [71]:
# CELL 9 — DimTime, DimLocalAuthority và DimLocation

all_results.append(
    profile_standard_dimension(
        dimension_name="DimTime",
        dataframe=df_col,
        grain_columns=["collision_time"],
        source_file="collisions_clean.csv",
        note="Một dòng cho một phút trong ngày.",
    )
)

all_results.append(
    profile_standard_dimension(
        dimension_name="DimLocalAuthority",
        dataframe=df_col,
        grain_columns=[
            "local_authority_district_code"
        ],
        source_file="collisions_clean.csv",
        note="Một dòng cho một Local Authority.",
    )
)

all_results.append(
    profile_standard_dimension(
        dimension_name="DimLocation",
        dataframe=df_col,
        grain_columns=[
            "longitude",
            "latitude",
            "local_authority_district_code",
        ],
        source_file="collisions_clean.csv",
        note=(
            "Giữ các location thiếu một phần thông tin; "
            "chỉ tổ hợp toàn Unknown dùng LocationKey = 0."
        ),
    )
)

display(pd.DataFrame(all_results[-3:]))


,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimTime,collisions_clean.csv,collision_time,513801,1440,0,0,1440,1,1441,Một dòng cho một phút trong ngày.
1,DimLocalAuthority,collisions_clean.csv,local_authority_district_code,513801,370,0,0,370,1,371,Một dòng cho một Local Authority.
2,DimLocation,collisions_clean.csv,"longitude, latitude, local_authority_district_...",513801,512359,33,0,512359,1,512360,Giữ các location thiếu một phần thông tin; chỉ...


In [72]:
# CELL 10 — DimRoad, DimEnvironment và DimJunction

all_results.append(
    profile_standard_dimension(
        dimension_name="DimRoad",
        dataframe=df_col,
        grain_columns=[
            "first_road_class",
            "road_type",
            "speed_limit_mph",
            "area_type",
        ],
        source_file="collisions_clean.csv",
        note=(
            "Một tổ hợp Road Class, Road Type, "
            "Speed Limit và Area Type."
        ),
    )
)

all_results.append(
    profile_standard_dimension(
        dimension_name="DimEnvironment",
        dataframe=df_col,
        grain_columns=[
            "light_conditions",
            "weather_conditions",
            "road_surface_conditions",
        ],
        source_file="collisions_clean.csv",
        note=(
            "Tổ hợp toàn Unknown được đại diện "
            "bởi EnvironmentKey = 0."
        ),
    )
)

all_results.append(
    profile_standard_dimension(
        dimension_name="DimJunction",
        dataframe=df_col,
        grain_columns=[
            "junction_detail",
            "junction_control",
            "pedestrian_crossing",
        ],
        source_file="collisions_clean.csv",
        note=(
            "Tổ hợp toàn Unknown được đại diện "
            "bởi JunctionKey = 0."
        ),
    )
)

display(pd.DataFrame(all_results[-3:]))

,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimRoad,collisions_clean.csv,"first_road_class, road_type, speed_limit_mph, ...",513801,385,81,0,385,1,386,"Một tổ hợp Road Class, Road Type, Speed Limit ..."
1,DimEnvironment,collisions_clean.csv,"light_conditions, weather_conditions, road_sur...",513801,228,87,1,227,1,228,Tổ hợp toàn Unknown được đại diện bởi Environm...
2,DimJunction,collisions_clean.csv,"junction_detail, junction_control, pedestrian_...",513801,295,115,1,294,1,295,Tổ hợp toàn Unknown được đại diện bởi Junction...


In [73]:

# CELL 11 — DimSeverity 

severity_values = pd.concat(
    [
        df_col[
            ["collision_severity"]
        ].rename(
            columns={
                "collision_severity": "severity_name"
            }
        ),
        df_cas[
            ["casualty_severity"]
        ].rename(
            columns={
                "casualty_severity": "severity_name"
            }
        ),
    ],
    ignore_index=True,
)

severity_values["severity_name"] = (
    canonicalize_unknown(
        severity_values["severity_name"]
    )
)

result = profile_standard_dimension(
    dimension_name="DimSeverity",
    dataframe=severity_values,
    grain_columns=["severity_name"],
    source_file=(
        "collisions_clean.csv + casualties_clean.csv"
    ),
    note=(
        "Role-playing Dimension dùng cho Collision "
        "Severity và Casualty Severity."
    ),
)

all_results.append(result)
display(pd.DataFrame([result]))


,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimSeverity,collisions_clean.csv + casualties_clean.csv,severity_name,1166622,3,0,0,3,1,4,Role-playing Dimension dùng cho Collision Seve...


In [74]:
# CELL 12 — DimPedestrianContext

pedestrian_mask = (
    df_cas["casualty_class"]
    .str.strip()
    .str.casefold()
    .eq("pedestrian")
)

df_pedestrian = df_cas.loc[
    pedestrian_mask,
    [
        "pedestrian_location",
        "pedestrian_movement",
    ],
].copy()

# Nếu casualty là Pedestrian nhưng thuộc tính lại ghi
# "Not a Pedestrian" thì coi đây là Unknown.
for column in [
    "pedestrian_location",
    "pedestrian_movement",
]:
    not_pedestrian_mask = (
        df_pedestrian[column]
        .str.strip()
        .str.casefold()
        .eq("not a pedestrian")
    )

    df_pedestrian.loc[
        not_pedestrian_mask,
        column,
    ] = "Unknown"

    df_pedestrian[column] = canonicalize_unknown(
        df_pedestrian[column]
    )

pedestrian_result = profile_standard_dimension(
    dimension_name="DimPedestrianContext",
    dataframe=df_pedestrian,
    grain_columns=[
        "pedestrian_location",
        "pedestrian_movement",
    ],
    source_file="casualties_clean.csv",
    note=(
        "Chỉ đếm casualty là Pedestrian; "
        "có thêm Unknown và Not Applicable."
    ),
)

# profile_standard_dimension đã thêm Unknown.
# Bổ sung thêm một dòng Not Applicable.
pedestrian_result["Technical Members"] += 1
pedestrian_result["Expected Rows"] += 1

all_results.append(pedestrian_result)

print(
    f"Số casualty là Pedestrian: "
    f"{len(df_pedestrian):,}"
)

display(pd.DataFrame([pedestrian_result]))


Số casualty là Pedestrian: 94,398


,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimPedestrianContext,casualties_clean.csv,"pedestrian_location, pedestrian_movement",94398,84,17,1,83,2,85,Chỉ đếm casualty là Pedestrian; có thêm Unknow...


In [75]:
# CELL 13 — Tạo logical PedestrianContext cho Casualty Profile

df_cas_profile = df_cas[
    [
        "casualty_age_band",
        "casualty_sex",
        "casualty_class",
        "casualty_type",
        "pedestrian_location",
        "pedestrian_movement",
    ]
].copy()

for column in [
    "casualty_age_band",
    "casualty_sex",
    "casualty_class",
    "casualty_type",
    "pedestrian_location",
    "pedestrian_movement",
]:
    df_cas_profile[column] = canonicalize_unknown(
        df_cas_profile[column]
    )

is_pedestrian = (
    df_cas_profile["casualty_class"]
    .str.casefold()
    .eq("pedestrian")
)

# Mặc định: casualty không phải pedestrian
df_cas_profile["pedestrian_context"] = (
    "Not Applicable"
)

# Sửa các giá trị bất hợp lý trong pedestrian rows
for column in [
    "pedestrian_location",
    "pedestrian_movement",
]:
    invalid_not_pedestrian = (
        is_pedestrian
        & df_cas_profile[column]
            .str.casefold()
            .eq("not a pedestrian")
    )

    df_cas_profile.loc[
        invalid_not_pedestrian,
        column,
    ] = "Unknown"

pedestrian_location_unknown = is_unknown(
    df_cas_profile["pedestrian_location"]
)

pedestrian_movement_unknown = is_unknown(
    df_cas_profile["pedestrian_movement"]
)

both_pedestrian_fields_unknown = (
    pedestrian_location_unknown
    & pedestrian_movement_unknown
)

# Pedestrian nhưng thiếu toàn bộ context
df_cas_profile.loc[
    is_pedestrian
    & both_pedestrian_fields_unknown,
    "pedestrian_context",
] = "Unknown"

# Pedestrian có ít nhất một thông tin context
has_pedestrian_context = (
    is_pedestrian
    & ~both_pedestrian_fields_unknown
)

df_cas_profile.loc[
    has_pedestrian_context,
    "pedestrian_context",
] = (
    df_cas_profile.loc[
        has_pedestrian_context,
        "pedestrian_location",
    ]
    + " | "
    + df_cas_profile.loc[
        has_pedestrian_context,
        "pedestrian_movement",
    ]
)

display(
    df_cas_profile[
        [
            "casualty_class",
            "pedestrian_location",
            "pedestrian_movement",
            "pedestrian_context",
        ]
    ].head(10)
)


,casualty_class,pedestrian_location,pedestrian_movement,pedestrian_context
0,Pedestrian,"In centre of carriageway - not on refuge, isla...",Crossing from driver's nearside,"In centre of carriageway - not on refuge, isla..."
1,Pedestrian,"In carriageway, crossing elsewhere",Crossing from driver's nearside,"In carriageway, crossing elsewhere | Crossing ..."
2,Pedestrian,"In carriageway, crossing elsewhere",Unknown,"In carriageway, crossing elsewhere | Unknown"
3,Pedestrian,On footway or verge,Unknown,On footway or verge | Unknown
4,Pedestrian,Crossing elsewhere within 50m. of pedestrian c...,Crossing from driver's nearside,Crossing elsewhere within 50m. of pedestrian c...
5,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's offside,Crossing on pedestrian crossing facility | Cro...
6,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's nearside,Crossing on pedestrian crossing facility | Cro...
7,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's nearside,Crossing on pedestrian crossing facility | Cro...
8,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's offside,Crossing on pedestrian crossing facility | Cro...
9,Pedestrian,"In carriageway, crossing elsewhere",Crossing from driver's offside,"In carriageway, crossing elsewhere | Crossing ..."


In [76]:
# CELL 14 — DimCasualtyProfile

casualty_profile_result = (
    profile_standard_dimension(
        dimension_name="DimCasualtyProfile",
        dataframe=df_cas_profile,
        grain_columns=[
            "casualty_age_band",
            "casualty_sex",
            "casualty_class",
            "casualty_type",
            "pedestrian_context",
        ],
        source_file="casualties_clean.csv",
        note=(
            "Pedestrian context đã được chuẩn hóa thành "
            "Unknown, Not Applicable hoặc context cụ thể."
        ),
    )
)

all_results.append(casualty_profile_result)
display(pd.DataFrame([casualty_profile_result]))


,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimCasualtyProfile,casualties_clean.csv,"casualty_age_band, casualty_sex, casualty_clas...",652821,2455,644,0,2455,1,2456,Pedestrian context đã được chuẩn hóa thành Unk...


In [78]:
# Cell 15 — Lọc vehicle thực sự được Fact tham chiếu
fact_vehicle_keys = (
    df_cas[
        [
            "collision_id",
            "vehicle_reference",
        ]
    ]
    .drop_duplicates()
)

df_fact_vehicle = df_veh.merge(
    fact_vehicle_keys,
    on=[
        "collision_id",
        "vehicle_reference",
    ],
    how="inner",
    validate="one_to_one",
)

print("VEHICLE COVERAGE")
print(
    f"Toàn bộ vehicle nguồn : "
    f"{len(df_veh):,}"
)
print(
    f"Vehicle được Fact dùng: "
    f"{len(df_fact_vehicle):,}"
)
print(
    f"Vehicle không vào Fact : "
    f"{len(df_veh) - len(df_fact_vehicle):,}"
)


VEHICLE COVERAGE
Toàn bộ vehicle nguồn : 937,265
Vehicle được Fact dùng: 566,945
Vehicle không vào Fact : 370,320


In [ ]:
# Cell 16 — DimVehicleProfile
vehicle_profile_result = (
    profile_standard_dimension(
        dimension_name="DimVehicleProfile",
        dataframe=df_fact_vehicle,
        grain_columns=[
            "vehicle_type",
            "driver_age_band",
            "driver_sex",
            "vehicle_manoeuvre",
            "skidding_and_overturning",
            "vehicle_leaving_carriageway",
        ],
        source_file=(
            "vehicles_clean.csv filtered by "
            "FactCasualty vehicle keys"
        ),
        note=(
            "Chỉ sử dụng vehicle được casualty "
            "tham chiếu và xuất hiện trong Fact."
        ),
    )
)

all_results.append(vehicle_profile_result)
display(pd.DataFrame([vehicle_profile_result]))

,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimVehicleProfile,vehicles_clean.csv filtered by FactCasualty ve...,"vehicle_type, driver_age_band, driver_sex, veh...",566945,16971,10091,1,16970,1,16971,Chỉ sử dụng vehicle được casualty tham chiếu v...


In [80]:
# Cell 17 — Sắp xếp báo cáo
DIMENSION_ORDER = [
    "DimDate",
    "DimTimeBand",
    "DimTime",
    "DimLocalAuthority",
    "DimLocation",
    "DimRoad",
    "DimEnvironment",
    "DimJunction",
    "DimSeverity",
    "DimPedestrianContext",
    "DimCasualtyProfile",
    "DimVehicleProfile",
]

report_df = pd.DataFrame(all_results)

report_df["Dimension"] = pd.Categorical(
    report_df["Dimension"],
    categories=DIMENSION_ORDER,
    ordered=True,
)

report_df = (
    report_df
    .sort_values("Dimension")
    .reset_index(drop=True)
)

display(report_df)

,Dimension,Source File,Grain,Total Source Rows,Source Distinct Rows,Partial-Unknown Combinations,All-Unknown Combinations,Valid Members,Technical Members,Expected Rows,Note
0,DimDate,collisions_clean.csv,collision_date,513801,1826,0,0,1826,1,1827,Một dòng cho một ngày; có thêm Unknown Date.
1,DimTimeBand,Business-defined,TimeBandName,513801,7,0,0,7,1,8,7 khung giờ nghiệp vụ + 1 dòng Unknown.
2,DimTime,collisions_clean.csv,collision_time,513801,1440,0,0,1440,1,1441,Một dòng cho một phút trong ngày.
3,DimLocalAuthority,collisions_clean.csv,local_authority_district_code,513801,370,0,0,370,1,371,Một dòng cho một Local Authority.
4,DimLocation,collisions_clean.csv,"longitude, latitude, local_authority_district_...",513801,512359,33,0,512359,1,512360,Giữ các location thiếu một phần thông tin; chỉ...
5,DimRoad,collisions_clean.csv,"first_road_class, road_type, speed_limit_mph, ...",513801,385,81,0,385,1,386,"Một tổ hợp Road Class, Road Type, Speed Limit ..."
6,DimEnvironment,collisions_clean.csv,"light_conditions, weather_conditions, road_sur...",513801,228,87,1,227,1,228,Tổ hợp toàn Unknown được đại diện bởi Environm...
7,DimJunction,collisions_clean.csv,"junction_detail, junction_control, pedestrian_...",513801,295,115,1,294,1,295,Tổ hợp toàn Unknown được đại diện bởi Junction...
8,DimSeverity,collisions_clean.csv + casualties_clean.csv,severity_name,1166622,3,0,0,3,1,4,Role-playing Dimension dùng cho Collision Seve...
9,DimPedestrianContext,casualties_clean.csv,"pedestrian_location, pedestrian_movement",94398,84,17,1,83,2,85,Chỉ đếm casualty là Pedestrian; có thêm Unknow...


In [81]:
# Cell 18 — Đối chiếu kết quả mong đợi
EXPECTED_RESULTS = {
    "DimDate": 1827,
    "DimTimeBand": 8,
    "DimTime": 1441,
    "DimLocalAuthority": 371,
    "DimLocation": 512360,
    "DimRoad": 390,
    "DimEnvironment": 228,
    "DimJunction": 295,
    "DimSeverity": 4,
    "DimPedestrianContext": 85,
    "DimCasualtyProfile": 2456,
    "DimVehicleProfile": 17932,
}

validation_df = report_df[
    [
        "Dimension",
        "Expected Rows",
    ]
].copy()

validation_df["Reference Rows"] = (
    validation_df["Dimension"]
    .astype("object")
    .map(EXPECTED_RESULTS)
)

validation_df["Difference"] = (
    validation_df["Expected Rows"]
    - validation_df["Reference Rows"]
)

validation_df["Status"] = np.where(
    validation_df["Difference"] == 0,
    "PASS",
    "REVIEW",
)

display(validation_df)

failed = validation_df[
    validation_df["Status"] != "PASS"
]

if failed.empty:
    print(
        "TẤT CẢ DIMENSION ĐỀU KHỚP "
        "VỚI KẾT QUẢ MONG ĐỢI."
    )
else:
    print(
        "CÓ DIMENSION CẦN REVIEW. "
        "Không nên ghi số chính thức trước khi kiểm tra."
    )

,Dimension,Expected Rows,Reference Rows,Difference,Status
0,DimDate,1827,1827,0,PASS
1,DimTimeBand,8,8,0,PASS
2,DimTime,1441,1441,0,PASS
3,DimLocalAuthority,371,371,0,PASS
4,DimLocation,512360,512360,0,PASS
5,DimRoad,386,390,-4,REVIEW
6,DimEnvironment,228,228,0,PASS
7,DimJunction,295,295,0,PASS
8,DimSeverity,4,4,0,PASS
9,DimPedestrianContext,85,85,0,PASS


CÓ DIMENSION CẦN REVIEW. Không nên ghi số chính thức trước khi kiểm tra.


In [83]:
# Cell 19 — Xuất Excel
summary_df = report_df[
    [
        "Dimension",
        "Source Distinct Rows",
        "All-Unknown Combinations",
        "Valid Members",
        "Technical Members",
        "Expected Rows",
    ]
].copy()

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    report_df.to_excel(
        writer,
        sheet_name="Full Detail",
        index=False,
    )

    validation_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False,
    )

    for sheet_name in [
        "Summary",
        "Full Detail",
        "Validation",
    ]:
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

    summary_sheet = writer.sheets["Summary"]
    summary_sheet.column_dimensions["A"].width = 25
    summary_sheet.column_dimensions["B"].width = 22
    summary_sheet.column_dimensions["C"].width = 28
    summary_sheet.column_dimensions["D"].width = 18
    summary_sheet.column_dimensions["E"].width = 20
    summary_sheet.column_dimensions["F"].width = 18

    detail_sheet = writer.sheets["Full Detail"]
    detail_sheet.column_dimensions["A"].width = 25
    detail_sheet.column_dimensions["B"].width = 45
    detail_sheet.column_dimensions["C"].width = 75
    detail_sheet.column_dimensions["K"].width = 60

    validation_sheet = writer.sheets["Validation"]
    validation_sheet.column_dimensions["A"].width = 25
    validation_sheet.column_dimensions["B"].width = 18
    validation_sheet.column_dimensions["C"].width = 18
    validation_sheet.column_dimensions["D"].width = 15
    validation_sheet.column_dimensions["E"].width = 12

print(f"Đã xuất báo cáo: {OUTPUT_FILE}")

Đã xuất báo cáo: D:\OLAP\Data\Profiling\dimension_row_estimation_report.xlsx
